In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import pyodbc
from tkinter import font as tkfont

class CrimeRecordManager:
    def __init__(self, root):
        self.root = root
        self.root.title("Crime Record Management System")
        self.root.geometry("1200x800")
        self.root.configure(bg='#f0f0f0')
        
        # Create style for ttk widgets
        self.style = ttk.Style()
        self.style.configure('TButton', font=('Helvetica', 10))
        self.style.configure('TLabel', font=('Helvetica', 10), background='#f0f0f0')
        self.style.configure('Treeview', font=('Helvetica', 10))
        self.style.configure('Treeview.Heading', font=('Helvetica', 10, 'bold'))
        
        # Custom fonts for non-ttk widgets
        self.title_font = tkfont.Font(family="Helvetica", size=16, weight="bold")
        
        # Database connection
        self.conn = self.create_connection()
        self.cursor = self.conn.cursor()
        
        # Main container
        self.main_frame = tk.Frame(root, padx=20, pady=20, bg='#f0f0f0')
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title
        tk.Label(self.main_frame, text="Crime Record Management System", 
                font=self.title_font, bg='#f0f0f0').grid(row=0, column=0, columnspan=3, pady=10)
        
        # Search frame
        self.search_frame = tk.Frame(self.main_frame, bg='#f0f0f0')
        self.search_frame.grid(row=1, column=0, columnspan=3, sticky='ew', pady=5)
        
        # Table selection
        self.tables = ["Crime", "Victim", "Petitioner", "Criminal", "Evidence", "Arrested"]
        self.selected_table = tk.StringVar()
        self.selected_table.set(self.tables[0])
        
        ttk.Label(self.search_frame, text="Select Table:").pack(side=tk.LEFT, padx=5)
        self.table_dropdown = ttk.Combobox(self.search_frame, textvariable=self.selected_table, 
                                         values=self.tables, state="readonly", width=20)
        self.table_dropdown.pack(side=tk.LEFT, padx=5)
        self.table_dropdown.bind("<<ComboboxSelected>>", self.load_table_data)
        
        # Search entry
        self.search_var = tk.StringVar()
        self.search_entry = ttk.Entry(self.search_frame, textvariable=self.search_var, width=30)
        self.search_entry.pack(side=tk.LEFT, padx=5)
        self.search_entry.bind('<KeyRelease>', self.search_records)
        
        # Search button
        self.search_btn = ttk.Button(self.search_frame, text="Search", command=self.search_records)
        self.search_btn.pack(side=tk.LEFT, padx=5)
        
        # Treeview frame
        self.tree_frame = tk.Frame(self.main_frame, bg='#f0f0f0')
        self.tree_frame.grid(row=2, column=0, columnspan=3, sticky="nsew", pady=10)
        
        # Treeview with improved display
        self.tree = ttk.Treeview(self.tree_frame)
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        # Scrollbars
        v_scroll = ttk.Scrollbar(self.tree_frame, orient="vertical", command=self.tree.yview)
        v_scroll.pack(side=tk.RIGHT, fill=tk.Y)
        h_scroll = ttk.Scrollbar(self.tree_frame, orient="horizontal", command=self.tree.xview)
        h_scroll.pack(side=tk.BOTTOM, fill=tk.X)
        self.tree.configure(yscrollcommand=v_scroll.set, xscrollcommand=h_scroll.set)
        
        # Button frame
        self.btn_frame = tk.Frame(self.main_frame, bg='#f0f0f0')
        self.btn_frame.grid(row=3, column=0, columnspan=3, pady=10)
        
        # Buttons
        button_style = {'width': 12}
        ttk.Button(self.btn_frame, text="Add", command=self.open_add_dialog, **button_style).pack(side=tk.LEFT, padx=5)
        ttk.Button(self.btn_frame, text="Edit", command=self.open_edit_dialog, **button_style).pack(side=tk.LEFT, padx=5)
        ttk.Button(self.btn_frame, text="Delete", command=self.delete_record, **button_style).pack(side=tk.LEFT, padx=5)
        ttk.Button(self.btn_frame, text="Refresh", command=self.load_table_data, **button_style).pack(side=tk.LEFT, padx=5)
        ttk.Button(self.btn_frame, text="Exit", command=self.root.quit, **button_style).pack(side=tk.LEFT, padx=5)
        
        # Configure grid weights
        self.main_frame.grid_rowconfigure(2, weight=1)
        self.main_frame.grid_columnconfigure(0, weight=1)
        
        # Load initial data
        self.load_table_data()
    
    def create_connection(self):
        try:
            conn = pyodbc.connect(
                'DRIVER={SQL Server};'
                'SERVER=DESKTOP-RRUH018;'
                'DATABASE=jrnl;'
                'Trusted_Connection=yes;'
            )
            return conn
        except pyodbc.Error as e:
            messagebox.showerror("Database Error", f"Failed to connect to database: {str(e)}")
            self.root.quit()
    
    def load_table_data(self, event=None):
        table_name = self.selected_table.get()
        
        try:
            # Clear existing data
            self.tree.delete(*self.tree.get_children())
            
            # Get columns and their data types
            self.cursor.execute(f"""
                SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = '{table_name}'
                ORDER BY ORDINAL_POSITION
            """)
            columns_info = self.cursor.fetchall()
            
            # Configure tree columns
            columns = [col[0] for col in columns_info]
            self.tree["columns"] = columns
            self.tree.column("#0", width=0, stretch=tk.NO)  # Hide first empty column
            
            # Set column headings and widths with appropriate sizing
            for col_name, data_type, max_length in columns_info:
                self.tree.heading(col_name, text=col_name)
                
                # Set column width based on data type and content
                if data_type in ('varchar', 'char', 'nvarchar'):
                    width = min(200, max(80, (max_length or 20) * 8))  # Scale based on max length
                elif data_type in ('int', 'smallint', 'tinyint'):
                    width = 80
                elif data_type in ('date', 'datetime'):
                    width = 120
                else:
                    width = 100
                
                self.tree.column(col_name, width=width, anchor=tk.W)
            
            # Fetch and insert data with proper formatting
            self.cursor.execute(f"SELECT * FROM {table_name}")
            for row in self.cursor.fetchall():
                formatted_row = []
                for i, value in enumerate(row):
                    # Format dates and handle NULL values
                    if value is None:
                        formatted_row.append("NULL")
                    elif columns_info[i][1] in ('date', 'datetime'):
                        formatted_row.append(str(value))
                    else:
                        formatted_row.append(str(value))
                self.tree.insert("", tk.END, values=formatted_row)
                
        except Exception as e:
            messagebox.showerror("Database Error", f"Error loading {table_name}: {str(e)}")
    
    def search_records(self, event=None):
        table_name = self.selected_table.get()
        search_term = self.search_var.get().strip()
        
        if not search_term:
            self.load_table_data()
            return
        
        try:
            # Get column names
            self.cursor.execute(f"SELECT * FROM {table_name} WHERE 1=0")
            columns = [column[0] for column in self.cursor.description]
            
            # Build search query
            conditions = []
            params = []
            for col in columns:
                conditions.append(f"CONVERT(VARCHAR, {col}) LIKE ?")
                params.append(f"%{search_term}%")
            
            query = f"SELECT * FROM {table_name} WHERE {' OR '.join(conditions)}"
            
            # Clear existing data
            self.tree.delete(*self.tree.get_children())
            
            # Execute search
            self.cursor.execute(query, params)
            for row in self.cursor.fetchall():
                self.tree.insert("", tk.END, values=row)
                
        except Exception as e:
            messagebox.showerror("Search Error", f"Error searching records: {str(e)}")
    
    def open_add_dialog(self):
        table_name = self.selected_table.get()
        dialog = tk.Toplevel(self.root)
        dialog.title(f"Add New Record - {table_name}")
        dialog.grab_set()
        dialog.resizable(False, False)
        
        try:
            # Get columns and their properties
            self.cursor.execute(f"""
                SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = '{table_name}'
                ORDER BY ORDINAL_POSITION
            """)
            columns_info = self.cursor.fetchall()
            
            entries = {}
            for i, (col_name, data_type, nullable) in enumerate(columns_info):
                ttk.Label(dialog, text=f"{col_name} ({data_type})").grid(row=i, column=0, padx=5, pady=5, sticky='e')
                
                # Create appropriate input widgets based on data type
                if data_type in ('varchar', 'char', 'nvarchar', 'text'):
                    entry = ttk.Entry(dialog, width=30)
                elif data_type in ('int', 'smallint', 'tinyint'):
                    entry = ttk.Entry(dialog, width=15, validate='key')
                    entry['validatecommand'] = (entry.register(self.validate_int), '%P')
                elif data_type in ('date', 'datetime'):
                    entry = ttk.Entry(dialog, width=15)
                    ttk.Button(dialog, text="...", command=lambda e=entry: self.open_calendar(e)).grid(row=i, column=2)
                else:
                    entry = ttk.Entry(dialog, width=20)
                
                entry.grid(row=i, column=1, padx=5, pady=5)
                entries[col_name] = entry
            
            def save_record():
                try:
                    values = []
                    for col_name, data_type, nullable in columns_info:
                        value = entries[col_name].get().strip()
                        
                        # Validate required fields
                        if nullable == 'NO' and not value:
                            messagebox.showwarning("Validation Error", 
                                                 f"{col_name} is a required field")
                            entries[col_name].focus_set()
                            return
                            
                        # Convert data types
                        if data_type in ('int', 'smallint', 'tinyint'):
                            value = int(value) if value else None
                        values.append(value)
                    
                    # Build and execute insert query
                    placeholders = ", ".join(["?"] * len(values))
                    query = f"INSERT INTO {table_name} VALUES ({placeholders})"
                    self.cursor.execute(query, values)
                    self.conn.commit()
                    
                    messagebox.showinfo("Success", "Record added successfully")
                    dialog.destroy()
                    self.load_table_data()
                except ValueError:
                    messagebox.showerror("Error", "Please enter valid numeric values for numeric fields")
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to add record: {str(e)}")
            
            # Buttons
            btn_frame = tk.Frame(dialog)
            btn_frame.grid(row=len(columns_info)+1, column=0, columnspan=3, pady=10)
            
            ttk.Button(btn_frame, text="Save", command=save_record).pack(side=tk.LEFT, padx=5)
            ttk.Button(btn_frame, text="Cancel", command=dialog.destroy).pack(side=tk.LEFT, padx=5)
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load table structure: {str(e)}")
            dialog.destroy()
    
    def open_edit_dialog(self):
        table_name = self.selected_table.get()
        selected = self.tree.selection()
        
        if not selected:
            messagebox.showwarning("Warning", "Please select a record to edit")
            return
        
        try:
            # Get selected record data
            item = self.tree.item(selected[0])
            values = item['values']
            
            # Get columns and their properties
            self.cursor.execute(f"""
                SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = '{table_name}'
                ORDER BY ORDINAL_POSITION
            """)
            columns_info = self.cursor.fetchall()
            columns = [col[0] for col in columns_info]
            
            # Create dialog
            dialog = tk.Toplevel(self.root)
            dialog.title(f"Edit Record - {table_name}")
            dialog.grab_set()
            dialog.resizable(False, False)
            
            entries = {}
            for i, (col_name, data_type, nullable) in enumerate(columns_info):
                ttk.Label(dialog, text=f"{col_name} ({data_type})").grid(row=i, column=0, padx=5, pady=5, sticky='e')
                
                # Create appropriate input widgets based on data type
                if data_type in ('varchar', 'char', 'nvarchar', 'text'):
                    entry = ttk.Entry(dialog, width=30)
                elif data_type in ('int', 'smallint', 'tinyint'):
                    entry = ttk.Entry(dialog, width=15, validate='key')
                    entry['validatecommand'] = (entry.register(self.validate_int), '%P')
                elif data_type in ('date', 'datetime'):
                    entry = ttk.Entry(dialog, width=15)
                    ttk.Button(dialog, text="...", command=lambda e=entry: self.open_calendar(e)).grid(row=i, column=2)
                else:
                    entry = ttk.Entry(dialog, width=20)
                
                entry.grid(row=i, column=1, padx=5, pady=5)
                entry.insert(0, str(values[i]) if i < len(values) else "")
                entries[col_name] = entry
            
            def update_record():
                try:
                    # Get primary key column (first column)
                    pk_col = columns[0]
                    pk_value = values[0]
                    
                    # Build SET clause and values
                    set_clause = []
                    new_values = []
                    
                    for col_name, data_type, nullable in columns_info[1:]:  # Skip PK
                        value = entries[col_name].get().strip()
                        
                        # Validate required fields
                        if nullable == 'NO' and not value:
                            messagebox.showwarning("Validation Error", 
                                                 f"{col_name} is a required field")
                            entries[col_name].focus_set()
                            return
                            
                        # Convert data types
                        if data_type in ('int', 'smallint', 'tinyint'):
                            value = int(value) if value else None
                        
                        set_clause.append(f"{col_name} = ?")
                        new_values.append(value)
                    
                    query = f"UPDATE {table_name} SET {', '.join(set_clause)} WHERE {pk_col} = ?"
                    self.cursor.execute(query, (*new_values, pk_value))
                    self.conn.commit()
                    
                    messagebox.showinfo("Success", "Record updated successfully")
                    dialog.destroy()
                    self.load_table_data()
                except ValueError:
                    messagebox.showerror("Error", "Please enter valid numeric values for numeric fields")
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to update record: {str(e)}")
            
            # Buttons
            btn_frame = tk.Frame(dialog)
            btn_frame.grid(row=len(columns_info)+1, column=0, columnspan=3, pady=10)
            
            ttk.Button(btn_frame, text="Update", command=update_record).pack(side=tk.LEFT, padx=5)
            ttk.Button(btn_frame, text="Cancel", command=dialog.destroy).pack(side=tk.LEFT, padx=5)
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load record: {str(e)}")
    
    def delete_record(self):
        table_name = self.selected_table.get()
        selected = self.tree.selection()
        
        if not selected:
            messagebox.showwarning("Warning", "Please select a record to delete")
            return
        
        try:
            # Get selected record data
            item = self.tree.item(selected[0])
            values = item['values']
            
            # Get primary key column (first column)
            self.cursor.execute(f"""
                SELECT COLUMN_NAME 
                FROM INFORMATION_SCHEMA.COLUMNS 
                WHERE TABLE_NAME = '{table_name}' 
                AND ORDINAL_POSITION = 1
            """)
            pk_col = self.cursor.fetchone()[0]
            pk_value = values[0]
            
            # Confirm deletion
            if not messagebox.askyesno("Confirm Delete", 
                                     f"Are you sure you want to delete this record?\n{pk_col}: {pk_value}"):
                return
            
            # Execute delete
            self.cursor.execute(f"DELETE FROM {table_name} WHERE {pk_col} = ?", pk_value)
            self.conn.commit()
            
            messagebox.showinfo("Success", "Record deleted successfully")
            self.load_table_data()
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to delete record: {str(e)}")
    
    def validate_int(self, value):
        if value == "":
            return True
        try:
            int(value)
            return True
        except ValueError:
            return False
    
    def open_calendar(self, entry_widget):
        # Simplified calendar popup for date entry
        top = tk.Toplevel(self.root)
        top.title("Select Date")
        
        import calendar
        from datetime import datetime
        
        def set_date(day):
            entry_widget.delete(0, tk.END)
            entry_widget.insert(0, f"{datetime.now().year}-{datetime.now().month}-{day}")
            top.destroy()
        
        for i in range(1, 32):
            try:
                btn = ttk.Button(top, text=str(i), command=lambda d=i: set_date(d))
                btn.grid(row=(i-1)//7, column=(i-1)%7, padx=2, pady=2)
            except:
                break

if __name__ == "__main__":
    root = tk.Tk()
    app = CrimeRecordManager(root)
    root.mainloop()